### 02 - Feature Transformations and High/Low Risk Labeling

In [1]:
import pandas as pd
import numpy as np
import requests, io
import geopandas as gpd
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import os

In [2]:
DATA_PATH = "../../data/processed/finaldataset/preprocessed_consolidated_incidence.csv"

# Load monthly district-level data
preprocessed_consolidated_incidence = pd.read_csv(DATA_PATH)

print(preprocessed_consolidated_incidence.head())


  District  Month  Cases  Year YearMonth  Precipitation_mm  Temp_Min_C  \
0   Ampara      1    1.0  2008   2008-01        177.600000     19.8505   
1   Ampara      2    3.0  2008   2008-02        121.000000     18.9005   
2   Ampara      3    0.0  2008   2008-03        293.600000     21.8005   
3   Ampara      4    1.0  2008   2008-04        119.000010     23.0505   
4   Ampara      5    4.0  2008   2008-05         34.600002     24.3005   

   Temp_Max_C  Soil_Moisture_0_7cm  MRiceArea  ...      BIO11        BIO12  \
0     29.6005             0.380341    64490.0  ...  24.642167  1616.600007   
1     28.9005             0.362213    64490.0  ...  24.642167  1616.600007   
2     30.7505             0.385442    64490.0  ...  24.642167  1616.600007   
3     32.5505             0.359476    64490.0  ...  24.642167  1616.600007   
4     34.5505             0.269909    64490.0  ...  24.642167  1616.600007   

   BIO13      BIO14      BIO15  BIO16       BIO17       BIO18  BIO19  \
0  293.6  34.6

##### Lag Calculations

In [3]:
## sort by "District", "Year", "Month" to create lag variables
lepto = preprocessed_consolidated_incidence.sort_values(["District", "Year", "Month"]).reset_index(drop=True)

In [4]:

# Climate lag features (1-3 months)
climate_lag_cols = ["Precipitation_mm", "Temp_Min_C", "Temp_Max_C",
                    "Soil_Moisture_0_7cm"]
for col in climate_lag_cols:
    for lag in [1, 2, 3]:
        lepto[f"{col}_lag{lag}"] = lepto.groupby("District", observed=False)[col].shift(lag)

# Rolling statistics (shift first to prevent leakage)
for col in ["Precipitation_mm", "Soil_Moisture_0_7cm"]:
    shifted = lepto.groupby("District", observed=False)[col].shift(1)
    lepto[f"{col}_roll3_mean"] = shifted.rolling(3).mean().reset_index(level=0, drop=True)
    lepto[f"{col}_roll6_mean"] = shifted.rolling(6).mean().reset_index(level=0, drop=True)

# Cyclical month encoding
lepto["Month_sin"] = np.sin(2 * np.pi * lepto["Month"] / 12)
lepto["Month_cos"] = np.cos(2 * np.pi * lepto["Month"] / 12)

# Agriculture × climate interactions
lepto["MRice_x_Precip"] = lepto["MRiceArea"] * lepto["Precipitation_mm"]
lepto["SRice_x_Precip"] = lepto["SRiceArea"] * lepto["Precipitation_mm"]

# Temperature range
lepto["Temp_Range"] = lepto["Temp_Max_C"] - lepto["Temp_Min_C"]

# Population density proxy
lepto["Pop_per_Household"] = lepto["Population"] / (lepto["Households"] + 1)


# District-relative precipitation anomaly
# Compute norms from train portion only to avoid leakage
split_year        = 2020
train_mask        = lepto["Year"] <= split_year
district_precip_mean = lepto.loc[train_mask].groupby("District", observed=False)["Precipitation_mm"].mean()
monthly_precip_mean  = lepto.loc[train_mask].groupby("Month")["Precipitation_mm"].mean()

lepto["Precip_vs_norm"] = lepto["Precipitation_mm"] / lepto["District"].astype(str).map(district_precip_mean)
lepto["Precip_anomaly"] = lepto["Precipitation_mm"] - lepto["Month"].map(monthly_precip_mean)

# Waterlogging & heat moisture interaction
lepto["Waterlog_index"] = lepto["Soil_Moisture_0_7cm"] * lepto["Precipitation_mm"]
lepto["Heat_Moisture"]  = lepto["mean_temp"] * lepto["Precipitation_mm"]

# Impute lag/roll NaNs with district × month seasonal median

lag_cols_created = [c for c in lepto.columns if "lag" in c or "roll" in c]
# Compute medians from train set only
seasonal_medians = (
    lepto.loc[train_mask]
    .groupby(["District", "Month"], observed=False)[lag_cols_created]
    .median()
    .reset_index()
)

district_medians = (
    lepto.loc[train_mask]
    .groupby("District", observed=False)[lag_cols_created]
    .median()
    .reset_index()
)

# Merge seasonal medians
lepto = lepto.merge(
    seasonal_medians,
    on=["District", "Month"],
    how="left",
    suffixes=("", "_seasonal")
)

# Merge district medians
lepto = lepto.merge(
    district_medians,
    on="District",
    how="left",
    suffixes=("", "_district")
)

# Fill NaNs
for col in lag_cols_created:
    lepto[col] = (
        lepto[col]
        .fillna(lepto[f"{col}_seasonal"])
        .fillna(lepto[f"{col}_district"])
    )


# Drop helper columns
cols_to_drop = [c for c in lepto.columns if "_seasonal" in c or "_district" in c]
lepto.drop(columns=cols_to_drop, inplace=True)

remaining_nans = lepto[lag_cols_created].isna().sum().sum()
print(f"Remaining NaNs after imputation: {remaining_nans}")


Remaining NaNs after imputation: 0


##### Risk Labeling


   - Training (2008-2020)
   - Testing (2021-2024)


In [5]:
# Use the monthly district-level data directly
lepto_monthly_incidence = lepto.copy()

# Year range check
min_year = lepto_monthly_incidence["Year"].min()
max_year = lepto_monthly_incidence["Year"].max()
print(f"Year range: {min_year}-{max_year}")

# Compute Monthly-Yearly Median Incidence (Relative Spatial Risk)
# Grouping by both Year and Month to calculate the baseline independently for each slice(year-month median for all districts)
monthly_medians = lepto_monthly_incidence.groupby(["Year", "Month"])["Incidence_per_100k"].transform("median")

# Assign RiskLabel depending on if a district exceeds the national median for that specific month-year
lepto_monthly_incidence["RiskLabel"] = (
    lepto_monthly_incidence["Incidence_per_100k"] > monthly_medians
    
).astype(int)


# District outbreak history
lepto_monthly_incidence["HR_lag1"]  = lepto_monthly_incidence.groupby("District", observed=False)["RiskLabel"].shift(1)
lepto_monthly_incidence["HR_lag2"]  = lepto_monthly_incidence.groupby("District", observed=False)["RiskLabel"].shift(2)
lepto_monthly_incidence["HR_roll3"] = (lepto_monthly_incidence.groupby("District", observed=False)["RiskLabel"]
                          .transform(lambda x: x.shift(1)
                                                 .rolling(3, min_periods=1)
                                                 .mean()))

# Impute NaNs (first month per district has no history)
# Train/test split 1st
train_end_year = 2020

lepto_monthly_train = lepto_monthly_incidence[
    lepto_monthly_incidence["Year"] <= train_end_year
].copy()

lepto_monthly_test = lepto_monthly_incidence[
    lepto_monthly_incidence["Year"] > train_end_year
].copy()

hr_cols = ["HR_lag1", "HR_lag2", "HR_roll3"]

district_medians = (lepto_monthly_train
                    .groupby("District", observed=False)[hr_cols]
                    .median()
)
for col in hr_cols:
    # Train
    lepto_monthly_train[col] = lepto_monthly_train[col].fillna(
        lepto_monthly_train["District"].map(district_medians[col])
    )
    
    # Test (use SAME medians)
    lepto_monthly_test[col] = lepto_monthly_test[col].fillna(
        lepto_monthly_test["District"].map(district_medians[col])
    )

print(f"Train set: {lepto_monthly_train.shape} | Years: {lepto_monthly_train['Year'].min()}-{lepto_monthly_train['Year'].max()}")
print(f"Test set:  {lepto_monthly_test.shape}  | Years: {lepto_monthly_test['Year'].min()}-{lepto_monthly_test['Year'].max()}")

Year range: 2008-2024
Train set: (3900, 66) | Years: 2008-2020
Test set:  (1200, 66)  | Years: 2021-2024


##### Prune redundant BIO fields

In [6]:

drop_cols = ["Year", "Cases", "Incidence_per_100k", "RiskLabel"]

X_tr   = lepto_monthly_train.drop(columns=drop_cols)
y_tr   = lepto_monthly_train["RiskLabel"]
X_test = lepto_monthly_test.drop(columns=drop_cols)
y_test = lepto_monthly_test["RiskLabel"]

# Drop BIOs that are semantically redundant with existing climate columns
always_drop_bio = ["BIO1", "BIO5", "BIO6", "BIO12"]

# Drop BIOs that are >0.90 correlated with another BIO
bio_cols = [c for c in X_tr.columns if c.startswith("BIO")]
bio_corr = X_tr[bio_cols].corr().abs()
upper    = bio_corr.where(np.triu(np.ones(bio_corr.shape), k=1).astype(bool))
auto_drop_bio = [col for col in upper.columns if any(upper[col] > 0.90)]

bio_to_drop = list(set(always_drop_bio + auto_drop_bio))
print(f"Dropping {len(bio_to_drop)} redundant BIO columns: {sorted(bio_to_drop)}")

X_tr   = X_tr.drop(columns=bio_to_drop, errors="ignore")
X_test = X_test.drop(columns=bio_to_drop, errors="ignore")

print(f"\nFinal Train : {X_tr.shape} | Class balance: {y_tr.value_counts().to_dict()}")
print(f"Final Test  : {X_test.shape} | Class balance: {y_test.value_counts().to_dict()}")


Dropping 9 redundant BIO columns: ['BIO1', 'BIO10', 'BIO11', 'BIO12', 'BIO5', 'BIO6', 'BIO7', 'BIO8', 'BIO9']

Final Train : (3900, 53) | Class balance: {0: 2028, 1: 1872}
Final Test  : (1200, 53) | Class balance: {0: 624, 1: 576}


##### Save dataset with risk labeling

In [7]:
## Combine
train_df = X_tr.copy()
train_df["RiskLabel"] = y_tr

test_df = X_test.copy()
test_df["RiskLabel"] = y_test

train_df.to_csv("../../data/processed/splitteddataset/lepto_monthly_train.csv", index=False)
test_df.to_csv("../../data/processed/splitteddataset/lepto_monthly_test.csv", index=False)

print("Train and Test datasets saved successfully.")

Train and Test datasets saved successfully.
